# Seasonality in CTA bus ridership

What ridership does over the course of a year, once the trend is removed and holiday weeks are
held out. Split out of `exploration.ipynb`, which this notebook depends on.

**Run `exploration.ipynb` and then `holidays.ipynb` first** — they write the three files read
below.

### Outline
0. Setup: load the cleaned daily data, the route inventory and the holiday calendar
1. Building a seasonal index
2. The pooled seasonal profile
3. How much do holiday weeks matter?
4. Is one system-wide profile enough?
5. Footnotes: the checks behind the choices in 1 and 2

## 0. Setup

Imports and plotting parameters are the same block as in `exploration.ipynb`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from numpy.lib.stride_tricks import sliding_window_view
from pathlib import Path

# Okabe-Ito: the published colour-vision-deficiency-safe categorical set.
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#009E73', '#CC79A7'
GRAY, INK = '#C9C9C9', '#333333'

# Ridership regimes. These are a COLOUR SCHEME for reading charts over time, not an
# analysis grouping: any analysis that needs a particular window defines it locally and
# prints what it used. 2020 gets its own colour because it is not comparable to anything
# else; the recovery years are lighter shades of it because the system has not returned
# to the pre-2020 level; 2025-present is distinct because the Frequent Network rollout
# begins 2025-03-23, so those years are not a clean baseline for anything.
ERAS = [('pre-2020',     2001, 2019, BLUE),
        ('2020',         2020, 2020, ORANGE),
        ('2021-2022',    2021, 2022, '#EE8A4E'),
        ('2023-2024',    2023, 2024, '#F5BE99'),
        ('2025-present', 2025, 2026, GREEN)]
ERA_ORDER = [e[0] for e in ERAS]
ERA_COLOR = {e[0]: e[3] for e in ERAS}

def era(year):
    """Map a calendar year to its ridership regime."""
    for name, lo, hi, _ in ERAS:
        if lo <= year <= hi:
            return name
    return None

# The 20 Frequent Network routes, as CTA labels them. Defined here because several
# sections need it.
FREQ = ['J14', '4', '9', '12', '20', '34', '47', '49', '53', '54',
        '55', '60', '63', '66', '72', '77', '79', '81', '82', '95']

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9A9A9A', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.8,
})
fmt_riders = FuncFormatter(lambda v, _: f'{v*1e-6:.1f}M' if v >= 1e6 else f'{v*1e-3:.0f}k')

In [ ]:
DERIVED = Path('data/derived')
for f in ('daily.csv', 'route_inventory.csv', 'holiday_calendar.csv'):
    if not (DERIVED / f).exists():
        raise FileNotFoundError(
            f'{DERIVED / f} is missing. Run exploration.ipynb (daily.csv, '
            'route_inventory.csv) and then holidays.ipynb (holiday_calendar.csv) first.')

d = pd.read_csv(DERIVED / 'daily.csv',
                dtype={'route': str, 'corridor': str},
                parse_dates=['date', 'week'])
inv = pd.read_csv(DERIVED / 'route_inventory.csv', dtype={'route': str}, index_col='route')
cal = pd.read_csv(DERIVED / 'holiday_calendar.csv', parse_dates=['date'])

print(f'daily rows        : {len(d):,}   routes {d.route.nunique()}')
print(f'route inventory   : {len(inv):,} routes')
print(f'holiday calendar  : {len(cal):,} dates, {int(cal.holiday.sum()):,} flagged as holidays')

# ---------------------------------------------------------------------------
# The one row-level exclusion made here: the 2013 Red Line South shuttles.
# They are a one-off construction artifact, not part of any seasonal cycle, and they
# sit inside a stretch of weeks that would otherwise look like a summer bump.
# ---------------------------------------------------------------------------
is_r = d.route.str.match(r'^R\d', na=False)
week_all = d.groupby('week').rides.sum()
week_cut = d[~is_r].groupby('week').rides.sum()
r_share = ((week_all - week_cut) / week_all).fillna(0)
affected = r_share > 0

print(f'\nR routes dropped  : {int(is_r.sum()):,} rows, '
      f'{", ".join(sorted(d.route[is_r].unique()))}')
print(f'  years they ran  : {sorted(set(d.date[is_r].dt.year))}')
print(f'  weeks affected  : {int(affected.sum())} '
      f'({week_all.index[affected].min().date()} .. {week_all.index[affected].max().date()})')
print(f'  their share of those weeks: mean {r_share[affected].mean()*100:.2f}%, '
      f'max {r_share[affected].max()*100:.2f}%')
print('  NOTE: this removes the shuttles only. Riders the shutdown pushed onto ordinary')
print('  routes (29, J14, 3, 4) are still in the totals, so the 2013 bump is reduced,')
print('  not removed.')

d = d[~is_r].copy()

# Weekly system totals, same construction as exploration.ipynb section 1.c.
wk = (d.groupby('week')
        .agg(rides=('rides', 'sum'), days=('date', 'nunique'), routes=('route', 'nunique'))
        .reset_index())
wk['partial'] = wk.days < 7
print(f'\nweeks             : {len(wk):,}   '
      f'{wk.week.min().date()} .. {wk.week.max().date()}   '
      f'partial (<7 days) {int(wk.partial.sum())}')

## 1. Building a seasonal index

Each week is divided by a **centred 2×52 moving average** — 53 weeks with the two end weeks at
half weight, which is centred and covers exactly one period. What is left is a multiplicative
*seasonal index*: 1.0 means "typical for this year", 0.9 means "10% below the year's own level".

Four choices go into that sentence. Each is checked in §5.

- **2×52, not a flat 53.** A centred window has to be odd, and a flat 53 covers one period plus
  a week. Half weights at the ends fix that: 2×52 returns a known seasonal amplitude to within
  1.1%, flat 53 to within 3.2%, flat 55 to within 5.1% (§5.c).
- **A moving average, not a fitted curve.** Both were tested against a known seasonal shape.
  Over 2001-2019 they are equally good; over the shorter 2022-2026 segment a degree-8
  polynomial has enough turning points to follow an annual cycle and eats 13% of the season,
  while the moving average stays within 1.1% (§5.b).
- **2020 and 2021 are dropped outright**, and the average never spans the gap. A window that
  straddles March 2020 has a denominator wrecked by the crash (§1 below).
- **Holiday weeks carry zero weight in the window and are held out of the profile.** The 152
  holiday dates come from `holidays.ipynb`. Christmas moves between ISO weeks 51, 52 and 1
  depending on the year, so a week-number average silently mixes holiday and normal weeks.

On the question that prompted all of this — whether a rising or falling trend tilts the year —
adding ±5%/yr of growth to the real series moves the recovered profile by 0.0013, against a
seasonal amplitude of 0.168 (§5.b).

In [ ]:
sw = wk[['week', 'rides', 'days', 'routes']].copy()
sw['year'] = sw.week.dt.year
sw['era']  = sw.year.map(era)
sw['woy']  = sw.week.dt.isocalendar().week.astype(int)     # ISO week number, 1..53

# A week is a holiday week if any of the 152 dates from holidays.ipynb falls inside it.
hol_mondays = (cal.loc[cal.holiday, 'date']
                  - pd.to_timedelta(cal.loc[cal.holiday, 'date'].dt.weekday, unit='D'))
sw['holiday_week'] = sw.week.isin(set(hol_mondays))

# The two windows the profile is built from. 2020-03-16 to 2021-12-27 is dropped: the
# shutdown and the recovery ramp. The moving average is computed inside each window
# separately so no window ever spans the gap -- a 53-week window straddling March 2020
# has a denominator halfway between two systems.
PRE_END  = pd.Timestamp('2020-03-09')     # last full week before the shutdown
POST_BEG = pd.Timestamp('2022-01-03')     # first week of 2022
SEGMENTS = {'2001-2019': sw.week <= PRE_END,
            '2022-2026': sw.week >= POST_BEG}
excluded = ~SEGMENTS['2001-2019'] & ~SEGMENTS['2022-2026']

print(f'excluded outright : {int(excluded.sum())} weeks, '
      f'{sw.week[excluded].min().date()} .. {sw.week[excluded].max().date()}')
for name, m in SEGMENTS.items():
    print(f'  segment {name}: {int(m.sum()):>4} weeks  '
          f'{sw.week[m].min().date()} .. {sw.week[m].max().date()}')

# 2x52 weights: 53 weeks, the two ends at half. Centred AND exactly one period wide.
WEIGHTS = np.ones(53)
WEIGHTS[0] = WEIGHTS[-1] = 0.5

def ma_2x52(values, keep):
    """Centred 2x52 moving average down each column of `values` (rows are weeks).

    `keep` is False for weeks that should carry no weight -- holiday weeks -- and NaNs
    are dropped the same way, with the remaining weights renormalised. The 26 weeks at
    each end have no full window and come back NaN. Accepts 1-D or 2-D input."""
    x = np.asarray(values, float)
    flat = x.ndim == 1
    if flat:
        x = x[:, None]
    w = np.where(keep[:, None] & ~np.isnan(x), 1.0, 0.0)
    num = np.einsum('ikj,j->ik', sliding_window_view(np.nan_to_num(x) * w, 53, axis=0), WEIGHTS)
    den = np.einsum('ikj,j->ik', sliding_window_view(w, 53, axis=0), WEIGHTS)
    out = np.full(x.shape, np.nan)
    out[26:-26] = np.where(den > 0, num / den, np.nan)
    return out[:, 0] if flat else out

sw['trend'] = np.nan
for name, m in SEGMENTS.items():
    rows = np.flatnonzero(m)
    sw.loc[rows, 'trend'] = ma_2x52(sw.rides.to_numpy()[rows],
                                    ~sw.holiday_week.to_numpy()[rows])
sw['index'] = sw.rides / sw.trend

usable = sw['index'].notna() & ~sw.holiday_week
print(f'\nweeks with a trend value : {int(sw["index"].notna().sum()):,} '
      f'(26 lost at each end of each segment)')
print(f'  holiday weeks removed  : {int((sw["index"].notna() & sw.holiday_week).sum())}')
print(f'  usable for the profile : {int(usable.sum()):,}')
for name, m in SEGMENTS.items():
    q = usable & m
    print(f'    {name}: {int(q.sum()):>3} weeks  '
          f'{sw.week[q].min().date()} .. {sw.week[q].max().date()}')
print(f'\n2026 contributes {int((usable & (sw.year == 2026)).sum())} weeks -- the last usable '
      f'week is {sw.week[usable].max().date()}.')
print('The Frequent Network window is not covered by this index and will need the profile')
print('extrapolated, not measured.')

In [ ]:
def profile(frame, col='index'):
    """Seasonal profile by ISO week number.

    The centre is the MEDIAN across years, not the mean: the year-to-year distribution is
    left-skewed (median skew -0.51, and 13 of 47 week-numbers skewed past |1.0|) because
    disruptions -- storms, cold snaps, shutdowns -- push a week down and nothing pushes it
    up. Both are reported in section 2 so the choice is visible. Spread is given two ways,
    the IQR to go with the median and the sd for anything downstream that needs one."""
    g = frame.groupby('woy')[col]
    return pd.DataFrame({'median': g.median(), 'mean': g.mean(),
                         'q25': g.quantile(.25), 'q75': g.quantile(.75),
                         'sd': g.std(), 'n': g.size()})

# By era first, as a diagnostic: the pooling decision in section 2 rests on these agreeing.
# 2020 and 2021 have no index at all now, so the 2020 line is empty and 2021-2022 is 2022 only.
profiles = {name: profile(sw[usable & (sw.era == name)]) for name in ERA_ORDER}

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.4), sharex=True)
for ax, dropped in zip(axes, ([], ['2021-2022'])):
    shown = [n for n in ERA_ORDER if n not in dropped and not profiles[n].empty]
    for name in shown:
        p = profiles[name]
        ax.plot(p.index, p['median'], lw=1.6, color=ERA_COLOR[name], label=name)
    ax.axhline(1, color=INK, lw=0.8, ls=':')
    ax.set_xlabel('ISO week of year')
    ax.set_title('every era with an index' if not dropped else 'without the 22 weeks of 2022',
                 loc='left', fontsize=10)
    ax.legend(frameon=False, ncol=2, loc='lower left', fontsize=8)
    lo = min(profiles[n]['median'].min() for n in shown)
    hi = max(profiles[n]['median'].max() for n in shown)
    print(f'{"all" if not dropped else "2021-2022 dropped":<20} median range {lo:.3f} .. {hi:.3f}')
axes[0].set_ylabel('seasonal index (1.0 = typical for that year)')
fig.suptitle('Seasonal profile by era — holiday weeks held out', x=0.005, ha='left', fontsize=11)
plt.tight_layout()
plt.show()

print('\nweek-numbers behind each era line:')
for name in ERA_ORDER:
    p = profiles[name]
    if p.empty:
        print(f'  {name:<14} no usable weeks')
        continue
    print(f'  {name:<14} {p.index.size:>2} week-numbers, {int(p.n.sum()):>4} weeks, '
          f'{int((p.n < 3).sum()):>2} of them resting on fewer than 3 observations')

In [ ]:
# Holding holiday weeks out leaves gaps. Some ISO week-numbers contain a holiday in every
# year, and worse, some survive in only one or two years -- those survivors are the odd
# years when the holiday fell in the neighbouring week, which is not a typical week.
cover = pd.DataFrame({'all_weeks': profile(sw[sw['index'].notna()])['n'],
                      'non_holiday': profile(sw[usable])['n']}).fillna(0).astype(int)
missing = [w for w in range(1, 54) if w not in cover.index[cover.non_holiday > 0]]

print(f'ISO week-numbers with no non-holiday observation at all: {missing}')
print('These need the holiday adjustment from holidays.ipynb, not this profile.\n')
print('thinnest coverage among the week-numbers that do survive:')
print(cover[cover.non_holiday > 0].nsmallest(8, 'non_holiday').to_string())
print('\nWeeks 22, 27 and 36 are Memorial Day, July 4th and Labor Day: the holiday lands in')
print('the same ISO week almost every year, so almost every observation is held out.')

## 2. The pooled seasonal profile

Every usable year pooled into one profile: the median index by ISO week, with the year-to-year
spread as the uncertainty.

The spread shown is the **variation between years**, not a confidence interval on the median.
That is deliberate. A CI says how well the average year is pinned down, which is not the
question when adjusting one specific week of one specific year — and it is narrow enough
(half-width ~0.013) to be badly misread. The IQR and sd say what an individual year does.

Pooling 2001-2019 with 2022-2025 needs the two to agree. They differ by 0.025 on average and
correlate 0.717, which sounds like disagreement until it is compared against how much two
ordinary blocks of pre-2020 years differ from each other — the same statistic runs 0.011 to
0.025 across 31 such blocks. Footnote §5.a is that calibration; without it the number below
means nothing.

In [ ]:
MIN_YEARS = 5           # below this a week-number is reported but not drawn as an estimate

pooled = profile(sw[usable])
pooled['iqr'] = pooled.q75 - pooled.q25
pooled['thin'] = pooled.n < MIN_YEARS
solid = pooled[~pooled.thin]

print(f'pooled over {int(pooled.n.sum())} weeks and {len(pooled)} week-numbers')
print(f'  median index  {pooled["median"].min():.3f} .. {pooled["median"].max():.3f}'
      f'   (amplitude {pooled["median"].max() - pooled["median"].min():.3f})')
print(f'\nyear-to-year spread, over the {len(solid)} week-numbers with at least {MIN_YEARS} years:')
print(f'  sd    median {solid.sd.median():.4f}   range {solid.sd.min():.4f} .. {solid.sd.max():.4f}')
print(f'  IQR   median {solid.iqr.median():.4f}   range {solid.iqr.min():.4f} .. {solid.iqr.max():.4f}')
print(f'  IQR / (1.349 x sd) = {(solid.iqr / (1.349 * solid.sd)).median():.3f}   '
      f'(1.0 would mean the year-to-year scatter is normal; below 1 means fat tails)')

# Median or mean? Reported rather than assumed.
dif = (solid['mean'] - solid['median'])
print(f'\nmean vs median centre: average difference {dif.mean():+.4f}, '
      f'largest {dif.abs().max():.4f} at week {dif.abs().idxmax()}, '
      f'{int((dif.abs() > 0.01).sum())} week-numbers differ by more than 0.01')
print(f'  amplitude {solid["median"].max() - solid["median"].min():.4f} on the median, '
      f'{solid["mean"].max() - solid["mean"].min():.4f} on the mean')

print(f'\nweek-numbers resting on fewer than {MIN_YEARS} years, drawn hollow: '
      f'{list(pooled.index[pooled.thin])}')
print('thinnest support:')
print(pooled.nsmallest(6, 'n')[['median', 'sd', 'n']].round(4).to_string())

fig, axes = plt.subplots(2, 1, figsize=(11.5, 7.2), sharex=True,
                         gridspec_kw={'height_ratios': [2.6, 1]})
ax = axes[0]
ax.fill_between(solid.index, solid.q25, solid.q75, color=BLUE, alpha=0.16, lw=0,
                label='IQR across years')
ax.plot(solid.index, solid['median'] + solid.sd, lw=1.0, ls='--', color=BLUE, alpha=.85,
        label='median ± 1 sd')
ax.plot(solid.index, solid['median'] - solid.sd, lw=1.0, ls='--', color=BLUE, alpha=.85)
ax.plot(solid.index, solid['median'], lw=1.9, color=BLUE, label='pooled median')
ax.scatter(pooled.index[pooled.thin], pooled.loc[pooled.thin, 'median'], s=26,
           facecolors='none', edgecolors=BLUE, lw=1.2, zorder=4,
           label=f'fewer than {MIN_YEARS} years — not an estimate')
ax.axhline(1, color=INK, lw=0.8, ls=':')
ax.set_ylabel('seasonal index')
ax.legend(frameon=False, ncol=2, loc='lower center', fontsize=8)
ax.set_title('Pooled seasonal profile — 2020 and 2021 excluded, holiday weeks held out, '
             '2×52 centred moving average', loc='left', fontsize=11)

ax = axes[1]
n_pre = sw[usable & SEGMENTS['2001-2019']].groupby('woy').size().reindex(pooled.index).fillna(0)
n_post = sw[usable & SEGMENTS['2022-2026']].groupby('woy').size().reindex(pooled.index).fillna(0)
ax.bar(pooled.index, n_pre, color=INK, alpha=.55, label='2001-2019')
ax.bar(pooled.index, n_post, bottom=n_pre, color=ORANGE, label='2022-2025')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('weeks contributing')
ax.legend(frameon=False, ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# The two halves, week by week -- the input to the pooling decision. Calibrated in 5.a.
a = profile(sw[usable & SEGMENTS['2001-2019']])['median']
b = profile(sw[usable & SEGMENTS['2022-2026']])['median']
shared = a.index.intersection(b.index)
print(f'2001-2019 vs 2022-2025 over {len(shared)} shared week-numbers:')
print(f'  correlation       {np.corrcoef(a[shared], b[shared])[0, 1]:.3f}')
print(f'  mean |difference| {(a[shared] - b[shared]).abs().mean():.4f}')
print(f'  largest           {(a[shared] - b[shared]).abs().max():.4f} '
      f'at week {(a[shared] - b[shared]).abs().idxmax()}')
print(f'  amplitude         {a.max() - a.min():.4f} (2001-2019) vs '
      f'{b.max() - b.min():.4f} (2022-2025)')

## 3. How much do holiday weeks matter?

The same profile computed with holiday weeks left in, against the one with them held out. The
gap is the reason they are held out rather than averaged over.

In [ ]:
# Holiday weeks carry no weight in the moving average either, so "included" here means
# putting them back into the profile only -- the baseline is identical in both lines.
with_hol = profile(sw[sw['index'].notna()])
without  = profile(sw[usable])
gap = (with_hol['median'] - without['median']).dropna()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(without.index, without['median'], lw=1.6, color=BLUE, label='holiday weeks held out')
ax.plot(with_hol.index, with_hol['median'], lw=1.3, color=ORANGE, ls='--',
        label='holiday weeks included')
ax.axhline(1, color=INK, lw=0.8, ls=':')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('seasonal index')
ax.set_title('Effect of leaving holiday weeks in the seasonal profile',
             loc='left', fontsize=11)
ax.legend(frameon=False)
plt.show()

print(f'week-numbers affected: {int((gap.abs() > 0.005).sum())} of {len(gap)} move by more than 0.005')
print(f'largest move {gap.abs().max():.3f} at week {gap.abs().idxmax()}, against a seasonal '
      f'amplitude of {without["median"].max() - without["median"].min():.3f}\n')
print('weeks most affected by including holiday weeks:')
print(gap.reindex(gap.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

## 4. Is one system-wide profile enough?

If routes have different seasonal shapes, a single system profile will mis-adjust individual
corridors. Each route's own profile is correlated against the system's.

In [ ]:
# Same baseline as the system profile: 2x52, per segment, holiday weeks at zero weight.
route_week = d.pivot_table(index='week', columns='route', values='rides', aggfunc='sum')
route_week = route_week.reindex(sw.week)                 # align to the weekly frame
keep = ~sw.holiday_week.to_numpy()

route_index = pd.DataFrame(np.nan, index=route_week.index, columns=route_week.columns)
for name, m in SEGMENTS.items():
    rows = np.flatnonzero(m.to_numpy())
    trend = ma_2x52(route_week.to_numpy()[rows], keep[rows])
    route_index.iloc[rows] = route_week.to_numpy()[rows] / trend

route_long = (route_index[keep].stack().rename('index').reset_index()
                .assign(woy=lambda t: t.week.dt.isocalendar().week.astype(int)))
print(f'route-weeks with an index: {len(route_long):,} across '
      f'{route_long.route.nunique()} routes')

sys_prof = pooled['median']
corrs, weeks_used = {}, {}
for r, grp in route_long.groupby('route'):
    p = grp.groupby('woy')['index'].median()
    common = p.index.intersection(sys_prof.index)
    weeks_used[r] = len(grp)
    corrs[r] = np.corrcoef(p[common], sys_prof[common])[0, 1] if len(common) > 20 else np.nan

cs = pd.Series(corrs).rename('corr_with_system')
print(f'\nroutes with a profile            : {int(cs.notna().sum())} of {len(cs)}')
print(f'  too few week-numbers to compare: {int(cs.isna().sum())}')
print(f'median correlation with system   : {cs.median():.3f}')
print(f'  quartiles                      : {cs.quantile(.25):.3f} .. {cs.quantile(.75):.3f}')
print(f'  routes below 0.5               : {int((cs < 0.5).sum())}')

print('\nleast like the system:')
print(pd.DataFrame({'corr': cs, 'name': inv.name, 'riders/day': inv['mean'].round(0),
                    'route_weeks': pd.Series(weeks_used)})
        .dropna(subset=['corr']).nsmallest(10, 'corr').round(3).to_string())
print('\nFrequent Network routes:')
print(pd.DataFrame({'corr': cs, 'name': inv.name})
        .reindex(FREQ).dropna(subset=['corr']).sort_values('corr').round(3).to_string())

## 5. Footnotes

Checks behind the choices made above. None of these feed the profile — they are the reason the
profile is built the way it is, and they are here so the choices can be argued with.

**a.** Is the gap between 2001-2019 and 2022-2025 bigger than the gap between two ordinary
blocks of years? This is the calibration behind pooling.

**b.** Moving average against a fitted polynomial, scored on a known seasonal shape. The
polynomial is not used here, but a fitted growth curve may be worth having in the Frequent
Network notebook as a separate estimate of system growth, so its behaviour is recorded.

**c.** Window width: 2×52 against the alternatives.

In [ ]:
# 5.a  Calibration of the pooling decision.
#
# 2001-2019 and 2022-2025 differ by some amount. On its own that number says nothing: two
# blocks of ordinary pre-2020 years also differ, and by how much is the benchmark. Every
# contiguous 3- and 4-year block of 2002-2019 is compared against the remaining pre-2020
# years by the identical procedure, giving the distribution the real comparison sits in.

def block_gap(block_years, ref_years):
    """Mean |difference| between two groups' profiles, over the week-numbers both cover
    with at least 2 and 5 observations respectively."""
    blk = profile(sw[usable & sw.year.isin(block_years)])
    ref = profile(sw[usable & sw.year.isin(ref_years)])
    common = blk.index[blk.n >= 2].intersection(ref.index[ref.n >= MIN_YEARS])
    return len(common), (blk.loc[common, 'median'] - ref.loc[common, 'median']).abs().mean()

PRE_YEARS = list(range(2002, 2020))
rows = []
for size in (3, 4):
    for i in range(len(PRE_YEARS) - size + 1):
        blk = PRE_YEARS[i:i + size]
        n, gap_ = block_gap(blk, [y for y in PRE_YEARS if y not in blk])
        rows.append({'size': size, 'block': f'{blk[0]}-{blk[-1]}', 'weeks': n, 'gap': gap_})
null = pd.DataFrame(rows)

post_years = sorted(sw.loc[usable & SEGMENTS['2022-2026'], 'year'].unique())
n_real, gap_real = block_gap(post_years, PRE_YEARS)
n_above = int((null.gap >= gap_real).sum())

for size in (3, 4):
    s = null[null['size'] == size]
    print(f'{size}-year blocks of ordinary years ({len(s)} blocks): mean |difference| '
          f'median {s.gap.median():.4f}, range {s.gap.min():.4f} .. {s.gap.max():.4f}')
print(f'\n{post_years[0]}-{post_years[-1]} against 2002-2019: mean |difference| {gap_real:.4f} '
      f'over {n_real} week-numbers')
print(f'  {n_above} of {len(null)} ordinary blocks differ from their own reference by at least '
      f'that much,')
print(f'  putting the real comparison at the {(1 - n_above / len(null)) * 100:.0f}th percentile '
      f'of the null. Larger than typical, not outside the range.')
print('\nthe five ordinary blocks that differ most from their own reference:')
print(null.nlargest(5, 'gap').round(4).to_string(index=False))

In [ ]:
# 5.b and 5.c  What a baseline is scored on.
#
# Real data cannot settle this: the true season is unknown, so a method that absorbs part of
# it looks the same as one that does not. So the truth is written down instead -- an analytic
# trend times a fixed seasonal shape, no noise -- and each baseline is asked to give the shape
# back. Two shapes are used because how much a baseline absorbs depends on how smooth the
# season is: a curve fitted over one year can follow a sine wave and cannot follow a plateau.
#
# Polynomials are fitted both to log rides and to rides, since the two are not equivalent and
# the difference is small enough to be worth measuring rather than assuming. See also the
# trend-invariance test in the next cell, where the two differ more clearly.

def recover(values, mask, baseline):
    """Seasonal profile of a synthetic series under a given baseline, normalised to mean 1."""
    rows = np.flatnonzero(mask.to_numpy() if hasattr(mask, 'to_numpy') else mask)
    base = baseline(values, rows)
    ix = pd.Series(np.asarray(values)[rows] / base, index=rows)
    ix = ix[~sw.holiday_week.to_numpy()[rows]].dropna()
    p = ix.groupby(sw.woy.to_numpy()[ix.index.to_numpy()]).median()
    return p / p.mean()

def as_ma(width, half_ends):
    def f(values, rows):
        w = np.ones(width)
        if half_ends:
            w[0] = w[-1] = 0.5
        x = np.asarray(values, float)[rows][:, None]
        k = np.where((~sw.holiday_week.to_numpy()[rows])[:, None], 1.0, 0.0)
        h = width // 2
        num = np.einsum('ikj,j->ik', sliding_window_view(x * k, width, axis=0), w)
        den = np.einsum('ikj,j->ik', sliding_window_view(k, width, axis=0), w)
        out = np.full((len(rows), 1), np.nan)
        out[h:-h] = np.where(den > 0, num / den, np.nan)
        return out[:, 0]
    return f

def as_poly(deg, space):
    def f(values, rows):
        t = ((sw.week.to_numpy()[rows] - sw.week.to_numpy()[rows][0])
             .astype('timedelta64[D]').astype(float) / 365.25)
        y = np.asarray(values, float)[rows]
        y = np.log(y) if space == 'log' else y
        k = ~sw.holiday_week.to_numpy()[rows]
        fit = np.polyval(np.polyfit(t[k], y[k], deg), t)
        return np.exp(fit) if space == 'log' else fit
    return f

woy_axis = np.arange(1, 54)
smooth = 1 + 0.06 * np.sin(2 * np.pi * (woy_axis - 8) / 52) + 0.025 * np.sin(4 * np.pi * (woy_axis - 3) / 52)
blocky = np.where(woy_axis < 9, 0.95, np.where(woy_axis < 34, 1.03, np.where(woy_axis < 46, 1.08, 1.0)))
SHAPES = {'smooth': pd.Series(smooth / smooth.mean(), index=woy_axis),
          'blocky': pd.Series(blocky / blocky.mean(), index=woy_axis)}

t_all = (sw.week - sw.week.min()).dt.days.to_numpy() / 365.25
TRENDS = {'2001-2019': np.exp(15.6 + 0.02 * t_all - 0.0022 * t_all ** 2),   # hump then decline
          '2022-2026': np.exp(15.4 - 0.55 * np.exp(-(t_all - 20.0) / 2.6))}  # decelerating rebound

BASELINES = {'2x52 (used here)':   as_ma(53, True),
             'flat 53':            as_ma(53, False),
             'flat 51':            as_ma(51, False),
             'flat 55':            as_ma(55, False),
             'poly deg 4, log':    as_poly(4, 'log'),
             'poly deg 4, level':  as_poly(4, 'level'),
             'poly deg 8, log':    as_poly(8, 'log'),
             'poly deg 8, level':  as_poly(8, 'level')}

print('Amplitude error: how much of the true seasonal amplitude survives. 0% is perfect,')
print('negative means the baseline absorbed part of the season into the trend.\n')
for shape_name, shape in SHAPES.items():
    print(f'{shape_name} season, true amplitude {shape.max() - shape.min():.4f}')
    print(f'  {"baseline":<20}' + ''.join(f'{k:>24}' for k in SEGMENTS))
    for lab, fn in BASELINES.items():
        cells = ''
        for seg, m in SEGMENTS.items():
            p = recover(TRENDS[seg] * sw.woy.map(shape).to_numpy(), m, fn)
            common = p.index.intersection(shape.index)
            true = shape[common] / shape[common].mean()
            cells += (f'{(p.max() - p.min()) / (shape.max() - shape.min()) * 100 - 100:>+11.1f}%'
                      f'  err {(p[common] - true).abs().max():.4f}')
        print(f'  {lab:<20}{cells}')
    print()

In [ ]:
# 5.b continued  Does the baseline absorb growth completely?
#
# The baseline is supposed to track the growth -- that is its whole job. The test is whether
# it captures ALL of it, leaving none behind in the seasonal profile. So: multiply the REAL
# weekly series by exp(g*t), which changes the trend and nothing about seasonality, and look
# at what comes out AFTER dividing by the refitted baseline. The baseline moves, by design.
# The profile should not. If it does, that growth ended up counted as season -- the failure
# this notebook started from: does a falling trend lift the start of the year above its true
# seasonal level?
#
# This is also where log and level fitting separate. Adding g*t to log rides is absorbed
# exactly by a polynomial fitted in log space, so its leftover profile is invariant by
# construction; a fit in levels is not. Neither is used to build the profile -- this is here
# because a fitted growth curve may be wanted in the Frequent Network notebook, and this says
# which to use.

print('Extra growth applied to the real series: exp(g*t), with the baseline refitted to the')
print('altered series. Movement in the recovered PROFILE is growth the baseline failed to')
print('absorb and left in the season. A baseline that took all of it scores 0.0000.\n')
print(f'  {"baseline":<20}{"segment":<12}{"g = -5%/yr":>12}{"g = +5%/yr":>12}{"tilt":>9}')
for lab in ('2x52 (used here)', 'poly deg 4, log', 'poly deg 4, level',
            'poly deg 8, log', 'poly deg 8, level'):
    fn = BASELINES[lab]
    for seg, m in SEGMENTS.items():
        base_p = recover(sw.rides.to_numpy(), m, fn)
        moves, tilts = [], []
        for g in (-0.05, 0.05):
            p = recover((sw.rides.to_numpy() * np.exp(g * t_all)), m, fn)
            common = p.index.intersection(base_p.index)
            moves.append((p[common] - base_p[common]).abs().max())
            tilts.append(np.polyfit(p.index, np.log(p), 1)[0] * 52)
        print(f'  {lab:<20}{seg:<12}{moves[0]:>12.4f}{moves[1]:>12.4f}{tilts[1] - tilts[0]:>9.4f}')
print('\n"tilt" is how far the profile leans when a 5%/yr decline is swapped for 5%/yr growth,')
print('in log points across the year, against a seasonal amplitude near 0.17.')

## Where this leaves us

### Established

- One pooled profile, 997 weeks over 51 week-numbers, median index 0.923–1.090 (§2).
- Year-to-year spread, median across week-numbers: sd 0.026, IQR 0.034. IQR / 1.349·sd = 0.94,
  so the year-to-year scatter has slightly fat tails (§2).
- Centre is the median, not the mean: the distribution is left-skewed (median skew −0.51),
  and the two centres differ by up to 0.041 at week 5 (§2).
- 2001-2019 and 2022-2025 differ by 0.0243 on average. Ordinary 3- and 4-year blocks of
  pre-2020 years differ from their own reference by 0.0110–0.0254 — the 97th percentile of
  that null, larger than typical but inside the range (§5.a).
- Adding ±5%/yr of growth to the real series moves the recovered profile by 0.0013 (§5.b).
- A 2×52 window recovers a known seasonal amplitude to within 1.1%; flat 53 to within 3.2%,
  flat 55 to within 5.1% (§5.c).
- A degree-8 polynomial absorbs 13% of a known season over the 2022-2026 segment; degree 4
  absorbs 2%. Log and level fits agree on shape, but only the log fit leaves no growth behind
  in the profile (§5.b).
- 2020-03-16 to 2021-12-27 excluded outright, 94 weeks. R shuttles excluded, 23 weeks
  affected, at most 3.7% of a week (§0).
- The index ends 2025-11-17. 2026 contributes nothing (§1).

### Open

1. **Holidays.** Weeks 52 and 53 have no non-holiday observation at all; weeks 1, 22, 27 and 36
   rest on 1–4 years. Holding out is not working for roughly six week-numbers a year, and the
   alternative is not obvious.
2. **Whether the profile is usable as it stands.** It is week-by-week and noisy; smoothing it,
   or fitting it, is a decision for when the adjustment is actually applied.
3. **The Frequent Network window has no measured index.** Anything applied there is
   extrapolated from earlier years.
4. Routes with inverted seasonality sit in the control group untreated (§4).